<a href="https://colab.research.google.com/github/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/Processing/NER_COLAB_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# מחברת NER חדשה ל־Google Colab

נבדקו כל ששת הקבצים המקומיים המבוקשים והמבנה ב־GitHub. המקור המקומי מכיל 428 מאמרים והוא זהה ל־`Data/DATA.json` ב־commit `abb6fd24a6446ad52e488f27a401f17a471c2efc`; SHA-256: `d58044161d4ec56f260852b9a773a9858b09438a97c00d8477c5dcc1101d71a2`.

במחברת הישנה יחידה הופרדה בשורות ריקות ו־Index נוצר לפני סינון. בחדשה כל שורה לא ריקה שאינה כותרת היא Text; רווחים נשמרים, ו־Index רציף מאפס אחרי הסינון. ב־Attack on Pearl Harbor נמצאו 147 יחידות, וכל 120 ה־Index הייחודיים שבדוגמה תואמים בדיוק. בדוגמת CSV יש 17 שגיאות word/offset בכלל 1,002 שורותיה; הן אינן משוכפלות כאן. אין להשוות Index ישן לחדש; יש להשוות מקור זהה והיסטי מסמך או חיתוך טקסט מאומת.

החוזה החדש הוא `ner-colab-3.2.1`: במקום זוג CSV/Parquet יחיד בגרסה 3.0, מופקים חמישה זוגות עצמאיים, אחד לכל מודל, כפי שהוגדר ב־3.1. ב־3.2 נוספו לוג התקנות מלא, אבחון תלויות ולוג זמני כל התאים ב־CSV/JSON עם מניפסט hashes משלו. תיקון 3.2.1 מתקין את Jedi החסרה בלוג Colab ומתאים את ספירת הטוקנים של spaCy לממשק Curated Transformers הקיים במודל 3.8.0; לא הוחלף מודל NER. ההגדרה TABLE_MODEL הוסרה. כל טבלה נגזרת רק מהאזכורים התקינים של המודל שלה, שומרת את שמונה עמודות הטמפלייט בסדר המקורי ומוסיפה subtype. המיפוי האחיד כולל את תוויות GLiNER, והתווית המקורית נשמרת בפלט המפורט וב־subtype. נשמרים גם המקור, המדגם, תוצאות המודלים הגולמיות, כל האזכורים והמועמדים, המניפסט והסיכום. אין בחירה של מודל זוכה; מטרת הטבלאות היא השוואה לפני בחירה מאוחרת. שדה table_rows בסיכום הוא מילון של שם מודל למספר שורות. הרצות קודמות נשמרות ללא שינוי. EVENT הוא אזכור NER בלבד, ללא הסקת עובדות/משתתפים, וההשוואה שלו נשארת לפי מודל. הדוגמה ו־Silver אינם קלט לחיזוי או לבחירה.

גרסאות הליבה תואמות למחברת הישנה; spaCy-transformers מוצמד ל־1.3.9 בטווח הישן, PEFT מוצמד ל־0.14.0 להתאמה ל־Transformers 4.49, ונוספו pandas/pyarrow ליצוא. Jedi מוצמד ל־0.19.2, שתומכת ב־Python 3.13. המתאמים spacy-curated-transformers==0.3.1 ו־curated-transformers==0.1.1 הם אותם רכיבים שהותקנו עם מודל spaCy בלוג שנשלח, ומוצמדים כעת במפורש. Torch נותר של Colab. פלט pip check המלא מוצג ונשמר. בדיקה נפרדת עוקבת אחר תלויות המחברת, כולל תלויות עקיפות ו־extras; סתירה בהן עוצרת, בעוד סתירות חיצוניות מדווחות כאזהרה ונשמרות. הלוג שנשלח זיהה Jedi חסרה כתלות היחידה שעצרה את בדיקת המחברת. אין כאן אימות של חיזוי כל המודלים ב־Colab. יש להשתמש ב־runtime חדש. חמשת המודלים גדולים ולכן נטענים בזה אחר זה; GPU מומלץ. spaCy מוגדר ל־CPU כמו במחברת הישנה כדי לא לדרוש CuPy. Flair פועל ללא הקשר בין חלונות בהגדרה גלויה. החפיפה היא במילים, תוך בדיקת מספר הטוקנים המקודד בפועל; חפיפה אינה מבטיחה שחזור אזכור שארוך מכל חלון.

הקוד נשען על API של [GLiNER2.5](https://huggingface.co/fastino/gliner2.5-base-v1) ועל [Flair 0.15.1](https://flairnlp.github.io/flair/v0.15.1/api/generated/flair.data.Sentence.html). התאים הסטטיים ובדיקות החוזה נבדקו מקומית; לא בוצעה כאן הרצת חמשת המודלים ב־Colab.



1 — התקנות
הריצו ב־runtime חדש. רק ספריית התקן משמשת לאתחול ההתקנות והטיימר; כל ייבוא חיצוני נמצא בתא הבא.

In [16]:
import sys
import subprocess
import time
import json
from contextlib import contextmanager

CELL_TIMINGS = globals().get("CELL_TIMINGS", [])
INSTALLATION_LOG = ""

def print_cell_timings():
    print("\nCELL TIMINGS — measured code blocks (logging/download bookkeeping excluded)")
    for row in CELL_TIMINGS:
        print(f"{row['title']} | {row['seconds']:.2f}s | {row['minutes']:.2f}min | {row['status']}")
    print(f"Total measured: {sum(row['seconds'] for row in CELL_TIMINGS):.2f}s")

@contextmanager
def cell(name):
    started = time.perf_counter()
    at_started = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    status, error = "success", None
    try:
        yield
    except BaseException as exc:
        status, error = "failed", f"{type(exc).__name__}: {exc}"
        raise
    finally:
        elapsed = time.perf_counter() - started
        CELL_TIMINGS.append({"attempt": len(CELL_TIMINGS)+1, "title": name,
            "seconds": elapsed, "minutes": elapsed/60, "status": status,
            "error": error, "at_started_utc": at_started,
            "at_finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})
        print(f"{'SUCCESS' if status == 'success' else 'FAILED'} {name}: "
              f"{error or 'completed'}; {elapsed:.2f}s ({elapsed/60:.2f}min)", flush=True)
        if status == "failed":
            print_cell_timings()

# Runs in a fresh Python process, so installed metadata is not cached by the kernel.
DEPENDENCY_CHECK_CODE = r"""
import json, sys
from importlib.metadata import distribution, PackageNotFoundError
from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet
from packaging.utils import canonicalize_name

def check_dependencies(roots, get_distribution=distribution):
    queue = [Requirement(value) for value in roots]
    visited, checked, errors = set(), set(), []
    while queue:
        req = queue.pop()
        name = canonicalize_name(req.name)
        try:
            dist = get_distribution(req.name)
        except PackageNotFoundError:
            errors.append(f"Missing dependency: {req}")
            continue
        if req.specifier and dist.version not in req.specifier:
            errors.append(f"{req}: installed {dist.version}")
        key = (name, tuple(sorted(req.extras)))
        if key in visited:
            continue
        visited.add(key)
        checked.add(name)
        requires_python = dist.metadata.get("Requires-Python")
        if requires_python and '.'.join(map(str, sys.version_info[:3])) not in SpecifierSet(requires_python):
            errors.append(f"{name} requires Python {requires_python}; actual {sys.version.split()[0]}")
        for value in dist.requires or []:
            child = Requirement(value)
            if child.marker is None or any(child.marker.evaluate({"extra": extra}) for extra in ["", *req.extras]):
                queue.append(child)
    return {"checked_packages": sorted(checked), "errors": sorted(set(errors))}

if __name__ == "__main__":
    report = check_dependencies(json.loads(sys.argv[1]))
    print(json.dumps(report))
    sys.exit(1 if report["errors"] else 0)
"""

with cell("1 — התקנות"):
    packages = [
        "transformers==4.49.0", "spacy>=3.8,<3.9",
        "spacy-transformers==1.3.9", "gliner2[local]==2.1.0",
        "spacy-curated-transformers==0.3.1", "curated-transformers==0.1.1",
        "flair==0.15.1", "huggingface-hub>=0.30,<1",
        "peft==0.14.0", "packaging>=24,<27", "pandas>=2.2,<3", "pyarrow>=18,<24",
        "jedi==0.19.2",  # IPython dependency missing from the reported Colab runtime
    ]
    def installation_command(args):
        global INSTALLATION_LOG
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        INSTALLATION_LOG += f"\nCOMMAND: {args}\n{result.stdout}\n"
        if result.stdout:
            print(result.stdout, flush=True)
        if result.returncode:
            raise RuntimeError(f"Installation command failed ({result.returncode}): {args}\n{result.stdout}")
    print("Python:", sys.version)
    installation_command([sys.executable, "-m", "pip", "install", "--quiet", *packages])
    installation_command([sys.executable, "-m", "spacy", "download", "en_core_web_trf"])
    full_check = subprocess.run([sys.executable, "-m", "pip", "check"], text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    INSTALLATION_LOG += "\nGLOBAL PIP CHECK:\n" + full_check.stdout
    print("GLOBAL PIP CHECK:\n" + full_check.stdout, flush=True)
    if full_check.returncode not in [0, 1]:
        raise RuntimeError(f"pip check itself failed: {full_check.stdout}")
    relevant = subprocess.run([sys.executable, "-c", DEPENDENCY_CHECK_CODE,
        json.dumps([*packages, "torch", "en-core-web-trf", "google-colab"])], text=True,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    INSTALLATION_LOG += "\nNOTEBOOK DEPENDENCY CHECK:\n" + relevant.stdout + relevant.stderr
    if relevant.returncode:
        raise RuntimeError("Notebook dependency conflict. Full details:\n" + relevant.stdout + relevant.stderr)
    DEPENDENCY_REPORT = {"global_pip_check_returncode": full_check.returncode,
        "global_pip_check_output": full_check.stdout, "notebook_dependencies": json.loads(relevant.stdout)}
    if full_check.returncode:
        print("WARNING: global conflicts were reported, but the notebook dependency closure passed. "
              "These diagnostics will be retained; model imports are checked in cell 2.")


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB ? eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

GLOBAL PIP CHECK:
gradio 6.29.0 has requirement huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2.
diffusers 0.40.0 has requirement huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2.

SUCCESS 1 — התקנות: completed; 69.22s (1.15min)


2 — ייבואים וגרסאות
בדיקת הגרסאות נכשלת בקול ברור אם הסביבה אינה תואמת. Torch נשאר מגרסת Colab כדי לשמור על התאמת CUDA.

In [17]:
with cell('2 — ייבואים וגרסאות'):
    import gc
    import hashlib
    import io
    import json
    import logging
    import math
    import platform
    import random
    import re
    import secrets
    import tempfile
    import zipfile
    from collections import Counter, defaultdict
    from datetime import datetime, timezone
    from importlib.metadata import version
    from pathlib import Path
    from zoneinfo import ZoneInfo

    import flair
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq
    import spacy
    import torch
    import transformers
    from flair.data import Sentence
    from flair.models import SequenceTagger
    from flair.splitter import SegtokSentenceSplitter
    from gliner2 import AutoExtractor
    from google.colab import files
    from huggingface_hub import hf_hub_download, model_info, snapshot_download
    from packaging.specifiers import SpecifierSet
    from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

    REQUIRED = {"transformers": "==4.49.0", "spacy": ">=3.8,<3.9",
                "spacy-transformers": "==1.3.9", "gliner2": "==2.1.0",
                "spacy-curated-transformers": "==0.3.1", "curated-transformers": "==0.1.1", "jedi": "==0.19.2",
                "flair": "==0.15.1", "huggingface-hub": ">=0.30,<1",
                "peft": "==0.14.0", "packaging": ">=24,<27", "pandas": ">=2.2,<3", "pyarrow": ">=18,<24"}
    VERSIONS = {name: version(name) for name in [*REQUIRED, "torch", "en-core-web-trf"]}
    for name, spec in REQUIRED.items():
        assert VERSIONS[name] in SpecifierSet(spec), (name, VERSIONS[name], spec)
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    log = logging.getLogger("ner_colab_v3")
    print(json.dumps(VERSIONS, indent=2))
    print("Python:", platform.python_version(), "CUDA:", torch.version.cuda)


{
  "transformers": "4.49.0",
  "spacy": "3.8.16",
  "spacy-transformers": "1.3.9",
  "gliner2": "2.1.0",
  "spacy-curated-transformers": "0.3.1",
  "curated-transformers": "0.1.1",
  "jedi": "0.19.2",
  "flair": "0.15.1",
  "huggingface-hub": "0.36.2",
  "peft": "0.14.0",
  "packaging": "26.3",
  "pandas": "2.2.3",
  "pyarrow": "23.0.1",
  "torch": "2.11.0+cu130",
  "en-core-web-trf": "3.8.0"
}
Python: 3.13.15 CUDA: 13.0
SUCCESS 2 — ייבואים וגרסאות: completed; 0.02s (0.00min)


3 — הגדרות מרכזיות
כל חמשת המודלים רצים ומקבלים טבלאות CSV/Parquet נפרדות על אותו מדגם. בחירת המודלים הזוכים תיעשה מאוחר יותר לאחר השוואה.

In [18]:
with cell('3 — הגדרות מרכזיות'):
    SAMPLE_SIZE = 1000
    SEED = 17
    RUN_RANDOM_ID = None  # אפשר להציב מספר שלם; אינו משפיע על המדגם
    OUTPUT_ROOT = Path("/content/TMP_NER")
    REPO_URL = "https://github.com/danielbr9382/Corpus2GeoRAG_Pacific.git"
    SOURCE_RELATIVE_PATH = "Data/DATA.json"
    SOURCE_COMMIT = None  # לשחזור: הציבו כאן את ה־SHA מהמניפסט הקודם
    MODELS = {
        "bert": {"id": "dslim/bert-base-NER", "revision": "d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc"},
        "roberta": {"id": "Jean-Baptiste/roberta-large-ner-english", "revision": None},
        "spacy": {"id": "en_core_web_trf", "revision": None},
        "gliner": {"id": "fastino/gliner2.5-base-v1", "revision": None},
        "flair": {"id": "flair/ner-english-ontonotes-large", "revision": None},
    }
    GLINER_LABELS = ["person", "organization", "military unit", "location",
        "geopolitical entity", "facility", "nationality or political group", "event",
        "date", "time", "weapon", "vehicle", "product", "work of art", "law",
        "language", "money", "quantity", "percent", "ordinal", "cardinal",
        "miscellaneous named entity"]
    GLINER_THRESHOLD = 0.5  # סף פענוח המקורי; כל מה שה־API מחזיר נשמר
    TABLE_THRESHOLDS = {name: None for name in MODELS}  # None = אין סינון נוסף
    EXPERIMENTAL_DATE = False  # Flair ראשי + GLiNER שאינו חופף לפי IoU
    DATE_OVERLAP_IOU = 0.5
    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    SPACY_DEVICE = "cpu"  # כמו במחברת הישנה; אינו דורש התקנת CuPy נוספת
    FLAIR_USE_CONTEXT = False  # מפורש: חיזוי חלון עצמאי, ללא הקשר מפסקאות אחרות
    WINDOWS = {name: {"max_encoded_tokens": 480, "max_words": 384, "overlap_words": 64}
               for name in MODELS}
    SCHEMA_VERSION = "ner-colab-3.2.1"
    SPLIT_VERSION = "nonempty_nonheading_lines_v1"
    assert isinstance(SAMPLE_SIZE, int) and SAMPLE_SIZE > 0
    assert isinstance(SEED, int)
    assert 0 <= DATE_OVERLAP_IOU <= 1
    if RUN_RANDOM_ID is None:
        RUN_RANDOM_ID = secrets.randbelow(900_000_000) + 100_000_000
    assert isinstance(RUN_RANDOM_ID, int) and RUN_RANDOM_ID >= 0
    for settings in WINDOWS.values():
        assert 0 <= settings["overlap_words"] < settings["max_words"]
        assert settings["max_encoded_tokens"] > 0
    YYYYMMDD = datetime.now(ZoneInfo("Asia/Jerusalem")).strftime("%Y%m%d")
    RUN_NAME = f"ner_seed{SEED}_n{SAMPLE_SIZE}_r{RUN_RANDOM_ID}_{YYYYMMDD}"
    RUN_DIR = OUTPUT_ROOT / RUN_NAME
    ZIP_PATH = OUTPUT_ROOT / f"{RUN_NAME}.zip"
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    if RUN_DIR.exists() or ZIP_PATH.exists():
        raise FileExistsError(f"Name collision: {RUN_NAME}. Change RUN_RANDOM_ID; no output overwritten.")
    RUN_DIR.mkdir(exist_ok=False)
    PATHS = {key: RUN_DIR / f"{key}_{RUN_NAME}.{ext}" for key, ext in {
        "source": "json", "sample": "json", "mentions_all": "json",
        "combination_experimental": "json", "validation_summary": "json",
        "manifest": "json", "selftests": "json", "environment": "txt", "installation_log": "txt",
    }.items()}
    PATHS.update({f"ner_{name}": RUN_DIR / f"ner_{name}_{RUN_NAME}.json" for name in MODELS})
    for name in MODELS:
        for extension in ["csv", "parquet"]:
            PATHS[f"entities_table_{name}_{extension}"] = RUN_DIR / f"entities_table_{name}_{RUN_NAME}.{extension}"
    TIMING_PATHS = {key: RUN_DIR / f"{key}_{RUN_NAME}.{extension}" for key, extension in {
        "cell_timings": "json", "cell_timings_table": "csv", "timing_manifest": "json"}.items()}
    CONFIG = {"schema_version": SCHEMA_VERSION, "split_version": SPLIT_VERSION,
        "sample_size": SAMPLE_SIZE, "seed": SEED, "run_random_id": RUN_RANDOM_ID,
        "date_jerusalem": YYYYMMDD, "repo_url": REPO_URL, "source_path": SOURCE_RELATIVE_PATH,
        "source_commit_requested": SOURCE_COMMIT, "models": MODELS, "windows": WINDOWS,
        "gliner_labels": GLINER_LABELS, "gliner_threshold": GLINER_THRESHOLD,
        "table_models": list(MODELS), "table_thresholds": TABLE_THRESHOLDS,
        "experimental_date": EXPERIMENTAL_DATE, "date_overlap_iou": DATE_OVERLAP_IOU,
        "device": DEVICE, "spacy_device": SPACY_DEVICE, "flair_use_context": FLAIR_USE_CONTEXT,
        "aggregation_strategy": "average", "prompt": None, "version_prompt": None}
    MODEL_OUTPUTS = {}
    print("Run:", RUN_NAME, "Device:", DEVICE, "Separate tables:", list(MODELS))


Run: ner_seed17_n1000_r915351843_20261013 Device: cuda:0 Separate tables: ['bert', 'roberta', 'spacy', 'gliner', 'flair']
SUCCESS 3 — הגדרות מרכזיות: completed; 0.02s (0.00min)


4 — כלי מקור, חלונות, היסטים ושמירה
החלונות נמדדים לפי הקידוד האמיתי של המודל, כולל סימנים מיוחדים וסכמת GLiNER. החפיפה נמדדת במילים ומוגדרת במפורש; שום טקסט אינו נחתך בשקט.

In [19]:
with cell('4 — כלי מקור, חלונות, היסטים ושמירה'):
    def utc_now():
        return datetime.now(timezone.utc).isoformat()

    def digest(payload):
        return hashlib.sha256(payload).hexdigest()

    def digest_file(path):
        hasher = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                hasher.update(block)
        return hasher.hexdigest()

    def jsonable(value):
        if hasattr(value, "item"):
            return value.item()
        raise TypeError(type(value).__name__)

    def encoded(obj):
        return json.dumps(obj, ensure_ascii=False, sort_keys=True, allow_nan=False,
                          default=jsonable).encode("utf-8")

    def save_json(path, obj):
        with Path(path).open("xb") as handle:
            handle.write(encoded(obj))

    def load_json(path):
        return json.loads(Path(path).read_bytes())

    def progress(name, done, total, errors, started, last):
        bucket = done * 10 // total
        if bucket > last:
            print(f"{name}: {100*done/total:.1f}% ({done}/{total}); errors={errors}; "
                  f"{time.perf_counter()-started:.1f}s", flush=True)
        return bucket

    def text_units(title, article):
        offset, index = 0, 0
        for line in article.splitlines(keepends=True):
            text = line.rstrip("\r\n")  # שומרים רווחים ותווים מקוריים
            if text.strip() and not re.fullmatch(r"=+\s*.*?\s*=+", text.strip()):
                end = offset + len(text)
                assert article[offset:end] == text
                yield {"Source": title, "Index": index, "Text": text,
                       "start_document": offset, "end_document": end}
                index += 1
            offset += len(line)

    def windows(text, settings, measure, limit):
        words = list(re.finditer(r"\S+", text))
        i = 0
        budget = min(settings["max_encoded_tokens"], limit)
        while i < len(words):
            left = 0 if i == 0 else words[i].start()
            high = min(len(words), i + settings["max_words"])
            def right(j):
                return words[j].start() if j < len(words) else len(text)
            # הפחתה עד שהקידוד המלא נכנס; לא מפעילים truncation
            j = high
            while j > i and measure(text[left:right(j)]) > budget:
                j -= max(1, (j-i)//8)
            if j == i:
                raise ValueError(f"Single word at {left} exceeds encoded token budget {budget}")
            end = right(j)
            count = measure(text[left:end])
            assert count <= budget
            yield {"start": left, "end": end, "Text": text[left:end], "encoded_tokens": count}
            if j == len(words):
                break
            i = max(i + 1, j - settings["overlap_words"])

    def collapse(text):
        normalized, starts, ends = [], [], []
        for match in re.finditer(r"\s+|\S", text):
            normalized.append(" " if match.group().isspace() else match.group())
            starts.append(match.start())
            ends.append(match.end())
        return "".join(normalized), starts, ends

    def group(label):
        value = re.sub(r"^[BISEUL]-", "", str(label)).lower()
        mapping = {"per": "PER", "person": "PER", "loc": "LOC", "gpe": "LOC",
            "fac": "LOC", "location": "LOC", "geopolitical entity": "LOC", "facility": "LOC",
            "org": "ORG", "organization": "ORG", "military unit": "ORG",
            "date": "DATE", "time": "TIME", "event": "EVENT"}
        return mapping.get(value, "MISC")

    def mention(p, name, prediction, base, ref):
        result = {"model": name, "model_id": MODELS[name]["id"],
            "Source": p["Source"], "Index": p["Index"], "sample_position": p["sample_position"],
            "Text": p["Text"], "source_sha256": SOURCE_META["sha256"],
            "source_commit": SOURCE_META["commit_sha"], "source_path": SOURCE_RELATIVE_PATH,
            "text_sha256": digest(p["Text"].encode()), "label_original": prediction.get("label"),
            "subtype": prediction.get("label"), "entity_group": group(prediction.get("label")),
            "score": None, "start": None, "end": None, "word": prediction.get("text"),
            "start_document": None, "end_document": None, "status_validation": "invalid",
            "error": None, "ref_output_raw": [ref], "prediction": prediction}
        try:
            score = prediction.get("score")
            if score is not None:
                score = float(score)
                if not math.isfinite(score):
                    raise ValueError("Non-finite score")
            result["score"] = score
            if prediction.get("alignment_error"):
                raise ValueError(prediction["alignment_error"])
            a, b = prediction["start"], prediction["end"]
            if not isinstance(a, int) or not isinstance(b, int) or isinstance(a, bool) or isinstance(b, bool):
                raise ValueError("Offsets must be integers")
            a, b = base + a, base + b
            result.update(start=a, end=b)
            if not 0 <= a < b <= len(p["Text"]):
                raise ValueError("Span outside paragraph")
            surface = p["Text"][a:b]
            # HF subword rendering is diagnostic only; other adapters verify model surface
            if prediction.get("surface_checked", True) and surface != prediction.get("text"):
                raise ValueError("Model surface differs from original source slice")
            da, db = p["start_document"] + a, p["start_document"] + b
            if DATA[p["Source"]][da:db] != surface:
                raise ValueError("Document slice differs from paragraph slice")
            if not isinstance(prediction.get("label"), str) or not prediction["label"]:
                raise ValueError("Missing label")
            result.update(word=surface, score=score, start_document=da, end_document=db,
                          status_validation="valid")
        except Exception as exc:
            result["error"] = f"{type(exc).__name__}: {exc}"
        return result

    def deduplicate(items):
        output, seen = [], {}
        for item in items:
            key = (item["Source"], item["Index"], item["model"], item["start"],
                   item["end"], item["label_original"])
            if item["status_validation"] == "valid" and key in seen:
                seen[key]["ref_output_raw"].extend(item["ref_output_raw"])
            else:
                output.append(item)
                if item["status_validation"] == "valid":
                    seen[key] = item  # בחירה ראשונה לפי סדר חלונות; לא לפי ממוצע ציון
        return output

    def process_paragraph(p, name, predict, measure, limit):
        started, raw, found, errors = time.perf_counter(), [], [], []
        try:
            for wi, w in enumerate(windows(p["Text"], WINDOWS[name], measure, limit)):
                record = {"window_id": wi, **w, "at_processed": utc_now(), "output": None,
                          "error": None, "status_processing": "pending"}
                raw.append(record)
                try:
                    output = predict(w["Text"])
                    record["output"] = output  # שומרים לפני פירוש/ולידציה
                    output_errors = output.get("errors", []) if isinstance(output, dict) else []
                    errors.extend(output_errors)
                    ref = f"{PATHS['ner_'+name].name}#/results/{p['sample_position']}/raw/{wi}/output"
                    predictions = adapt_output(name, output, w["Text"])
                    for ordinal, prediction in enumerate(predictions):
                        item = mention(p, name, prediction, w["start"], ref)
                        item["raw_ordinal"] = ordinal
                        found.append(item)
                    record["status_processing"] = "error" if output_errors else "success"
                    record["error"] = "; ".join(output_errors) or None
                except Exception as exc:
                    record["error"] = f"{type(exc).__name__}: {exc}"
                    record["status_processing"] = "error"
                    errors.append(record["error"])
                    log.error(json.dumps({"model": name, "sample_position": p["sample_position"],
                                          "window": wi, "error": record["error"]}))
        except Exception as exc:
            errors.append(f"{type(exc).__name__}: {exc}")
        found = deduplicate(found)
        invalid = [m for m in found if m["status_validation"] != "valid"]
        status = "error" if errors else "partial_invalid" if invalid else "success"
        return {"paragraph": p, "raw": raw, "mentions": found,
            "status_processing": status, "errors": errors + [m["error"] for m in invalid],
            "at_processed": utc_now(), "processing_seconds": time.perf_counter()-started}

    def run_model(name, predict, measure, limit, metadata):
        started, results, failures, last = time.perf_counter(), [], 0, 0
        for done, p in enumerate(SAMPLE, 1):
            result = process_paragraph(p, name, predict, measure, limit)
            results.append(result)
            failures += result["status_processing"] != "success"
            last = progress(name, done, len(SAMPLE), failures, started, last)
        metadata = {**metadata, "device": metadata.get("device", DEVICE), "labels": metadata["labels"],
                    "window": WINDOWS[name], "effective_encoded_limit": limit,
                    "threshold": GLINER_THRESHOLD if name == "gliner" else None,
                    "prompt": None, "version_prompt": None}
        output = {"schema_version": SCHEMA_VERSION, "run_name": RUN_NAME,
            "model_id": MODELS[name]["id"], "metadata": metadata,
            "library_versions": VERSIONS, "sample_sha256": SAMPLE_HASH,
            "source": SOURCE_META, "configuration_sha256": digest(encoded(metadata)),
            "at_processed": utc_now(), "processing_seconds": time.perf_counter()-started,
            "results": results}
        save_json(PATHS["ner_"+name], output)
        MODEL_OUTPUTS[name] = output

    def release():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def tokenizer_measure(tokenizer):
        return lambda text: len(tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])

    def spacy_runtime_adapter(nlp):
        trf = nlp.get_pipe("transformer").model
        if trf.has_ref("piece_encoder"):
            # en_core_web_trf 3.8.0 uses Curated Transformers, not the HF wrapper.
            piece_encoder = trf.get_ref("piece_encoder")
            pytorch_model = trf.get_ref("transformer").shims[0]._model
            limit = int(pytorch_model.curated_encoder.max_seq_len)
            def measure(text):
                pieces = piece_encoder.predict([nlp.make_doc(text)])[0]
                return int(pieces.dataXd.shape[0])  # Includes BOS/EOS; no truncation.
            backend = "spacy-curated-transformers"
        elif hasattr(trf, "tokenizer") and hasattr(trf, "transformer"):
            pytorch_model = trf.transformer
            limit = min(int(trf.tokenizer.model_max_length), int(pytorch_model.config.max_position_embeddings))
            measure = tokenizer_measure(trf.tokenizer)
            backend = "spacy-transformers"
        else:
            raise RuntimeError("Unsupported spaCy transformer interface; inspect the installed pipeline")
        if limit <= 0:
            raise ValueError(f"Invalid spaCy model token limit: {limit}")
        return measure, limit, str(next(pytorch_model.parameters()).device), backend

    def hf_revision(name):
        return model_info(MODELS[name]["id"], revision=MODELS[name]["revision"] or "main").sha

    def file_hashes(root):
        return {str(p.relative_to(root)): digest_file(p)
                for p in sorted(Path(root).rglob("*")) if p.is_file()}

    def adapt_output(name, output, text):
        if name in {"bert", "roberta"}:
            return [{"label": x["entity_group"], "start": int(x["start"]), "end": int(x["end"]),
                     "score": x.get("score"), "text": x.get("word"), "surface_checked": False}
                    for x in output]
        if name == "spacy":
            return output
        if name == "gliner":
            return [{"label": label, "start": x["start"], "end": x["end"],
                     "text": x["text"], "score": x.get("confidence")}
                    for label, values in output["entities"].items() for x in values]
        normalized, starts, ends = collapse(text)
        predictions, cursor = [], 0
        for record in output["sentences"] if isinstance(output, dict) else output:
            sentence_text = record["sentence_text"]
            s = record["sentence_start_flair"]
            # מיקום מדווח או מיקום הבא ברצף; אין חיפוש כללי של שם/משפט חוזר
            while cursor < len(normalized) and normalized[cursor].isspace():
                cursor += 1
            aligned = (s == cursor and normalized[s:s+len(sentence_text)] == sentence_text)
            cursor = s + len(sentence_text) if aligned else cursor
            for x in record["entities"]:
                a, b = s+x["start_pos"], s+x["end_pos"]
                valid = aligned and 0 <= a < b <= len(normalized)
                valid = valid and collapse(normalized[a:b])[0] == collapse(x["text"])[0]
                predictions.append({"label": x["label"], "score": x.get("score"),
                    "start": starts[a] if valid else None, "end": ends[b-1] if valid else None,
                    "text": text[starts[a]:ends[b-1]] if valid else x["text"],
                    "alignment_error": None if valid else "Flair sequential character alignment failed"})
        return predictions

    CONFIG_HASH = digest(encoded(CONFIG))
    print("Utilities ready; configuration SHA256:", CONFIG_HASH)


Utilities ready; configuration SHA256: 0bb24362b370945ecfd7824877da9dec7df3458f8e7d7153cad9e797f72c46b0
SUCCESS 4 — כלי מקור, חלונות, היסטים ושמירה: completed; 0.00s (0.00min)


5 — מקור GitHub ומדגם
Clone נפרד להרצה, בחירה מפורשת ב־Data/DATA.json ושמירת בתיו המקוריים. hash המדגם אינו כולל מזהה הרצה ולכן יציב באותו מקור, גודל וזרע.

In [20]:
with cell('5 — מקור GitHub ומדגם'):
    checkout = Path(tempfile.mkdtemp(prefix="ner_source_", dir="/content")) / "repo"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(checkout)], check=True)
    if SOURCE_COMMIT:
        if not re.fullmatch(r"[0-9a-fA-F]{40}", SOURCE_COMMIT):
            raise ValueError("SOURCE_COMMIT must be a full Git SHA")
        subprocess.run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", SOURCE_COMMIT], check=True)
        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"], check=True)
    commit = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
    tracked = subprocess.check_output(["git", "-C", str(checkout), "ls-tree", "-r", "--name-only", "HEAD"], text=True).splitlines()
    if SOURCE_RELATIVE_PATH not in tracked:
        raise FileNotFoundError(f"Missing tracked {SOURCE_RELATIVE_PATH}; JSON files: {[p for p in tracked if p.endswith('.json')]}")
    payload = (checkout / SOURCE_RELATIVE_PATH).read_bytes()
    DATA = json.loads(payload)
    if not isinstance(DATA, dict) or not DATA or not all(isinstance(k, str) and isinstance(v, str) for k, v in DATA.items()):
        raise ValueError("Expected a nonempty title -> full article text dictionary")
    SOURCE_META = {"repo_url": REPO_URL, "commit_sha": commit,
                   "path": SOURCE_RELATIVE_PATH, "sha256": digest(payload)}
    with PATHS["source"].open("xb") as handle:
        handle.write(payload)
    population, started, last = [], time.perf_counter(), 0
    for done, title in enumerate(sorted(DATA), 1):
        population.extend(text_units(title, DATA[title]))
        last = progress("paragraphs", done, len(DATA), 0, started, last)
    population.sort(key=lambda p: (p["Source"], p["start_document"], p["Index"]))
    if SAMPLE_SIZE > len(population):
        raise ValueError(f"Requested {SAMPLE_SIZE}, available {len(population)}")
    SAMPLE = random.Random(SEED).sample(population, SAMPLE_SIZE)
    SAMPLE = [{**p, "sample_position": i} for i, p in enumerate(SAMPLE)]
    for p in SAMPLE:
        assert DATA[p["Source"]][p["start_document"]:p["end_document"]] == p["Text"]
    save_json(PATHS["sample"], {"schema_version": SCHEMA_VERSION, "split_version": SPLIT_VERSION,
        "source": SOURCE_META, "seed": SEED, "sample_size": SAMPLE_SIZE, "paragraphs": SAMPLE})
    SAMPLE_HASH = digest(PATHS["sample"].read_bytes())
    with PATHS["environment"].open("x", encoding="utf-8") as handle:
        handle.write(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
    with PATHS["installation_log"].open("x", encoding="utf-8") as handle:
        handle.write(INSTALLATION_LOG)
    print("Source:", SOURCE_META, "Articles:", len(DATA), "Paragraphs:", len(population))
    print("Sample SHA256:", SAMPLE_HASH)


paragraphs: 10.0% (43/428); errors=0; 0.0s
paragraphs: 20.1% (86/428); errors=0; 0.1s
paragraphs: 30.1% (129/428); errors=0; 0.1s
paragraphs: 40.2% (172/428); errors=0; 0.1s
paragraphs: 50.0% (214/428); errors=0; 0.1s
paragraphs: 60.0% (257/428); errors=0; 0.1s
paragraphs: 70.1% (300/428); errors=0; 0.2s
paragraphs: 80.1% (343/428); errors=0; 0.2s
paragraphs: 90.2% (386/428); errors=0; 0.2s
paragraphs: 100.0% (428/428); errors=0; 0.2s
Source: {'repo_url': 'https://github.com/danielbr9382/Corpus2GeoRAG_Pacific.git', 'commit_sha': 'abb6fd24a6446ad52e488f27a401f17a471c2efc', 'path': 'Data/DATA.json', 'sha256': 'd58044161d4ec56f260852b9a773a9858b09438a97c00d8477c5dcc1101d71a2'} Articles: 428 Paragraphs: 36335
Sample SHA256: 533d457812ff859963721d00c48e5579bb5cf550b388070b0e4b30893c280a15
SUCCESS 5 — מקור GitHub ומדגם: completed; 16.33s (0.27min)


6 — בדיקות חוזה קצרות
בדיקות סינתטיות עצמאיות: חזרות, גבול חלון, מפת רווחים של Flair, תוויות, ציון חסר, פלט ריק וכשל שמור. בדיקת הקבצים עצמה תיערך גם בסוף.

In [21]:
with cell('6 — בדיקות חוזה קצרות'):
    original_data, original_meta = DATA, SOURCE_META
    DATA = {"test": "Paris. Paris. Battle, weapon."}
    SOURCE_META = {"sha256": "test", "commit_sha": "test"}
    p = {"Source": "test", "Index": 0, "Text": DATA["test"], "sample_position": 0,
         "start_document": 0, "end_document": len(DATA["test"])}
    try:
        repeated = [mention(p, "spacy", {"label": "GPE", "start": a, "end": a+5,
                        "text": "Paris", "score": None}, 0, "test") for a in [0, 7]]
        assert len(deduplicate(repeated)) == 2 and all(x["score"] is None for x in repeated)
        assert len(deduplicate([repeated[0], dict(repeated[0], ref_output_raw=["another"])])) == 1
        assert group("event") == "EVENT" and group("weapon") == "MISC"
        event = mention(p, "spacy", {"label": "EVENT", "start": 14, "end": 20,
                        "text": "Battle", "score": None}, 0, "test")
        misc = mention(p, "spacy", {"label": "weapon", "start": 22, "end": 28,
                       "text": "weapon", "score": None}, 0, "test")
        assert event["status_validation"] == misc["status_validation"] == "valid"
        assert event["entity_group"] == "EVENT" and misc["entity_group"] == "MISC"
        assert group("geopolitical entity") == "LOC" and group("military unit") == "ORG"
        bad = mention(p, "spacy", {"label": "EVENT", "start": 99, "end": 100,
                                  "text": "bad"}, 0, "test")
        assert bad["status_validation"] == "invalid" and bad["error"]
        empty = process_paragraph(p, "spacy", lambda text: [], len, 480)
        assert empty["status_processing"] == "success" and empty["mentions"] == []
        def fail(text):
            raise RuntimeError("intentional test failure")
        failed = process_paragraph(p, "spacy", fail, len, 480)
        assert failed["status_processing"] == "error" and failed["raw"][0]["error"]
        text = "a b c New York d e f g"
        ws = list(windows(text, {"max_encoded_tokens": 999, "max_words": 4, "overlap_words": 2}, len, 999))
        assert any("New York" in w["Text"] for w in ws)
        mapped = adapt_output("flair", [{"sentence_text": "Émile Zola.", "sentence_start_flair": 0,
            "entities": [{"label": "PERSON", "text": "Émile Zola", "start_pos": 0,
                          "end_pos": 10, "score": 0.9}]}], "Émile\t  Zola.")
        assert mapped[0]["text"] == "Émile\t  Zola" and mapped[0]["end"] == 12
        mapped = adapt_output("flair", [{"sentence_text": "Paris.", "sentence_start_flair": s,
            "entities": [{"label": "GPE", "text": "Paris", "start_pos": 0, "end_pos": 5}]}
            for s in [0, 7]], "Paris. Paris.")
        assert [x["start"] for x in mapped] == [0, 7]
        assert [x["Index"] for x in text_units("t", "\n== H ==\n x\ny\n")] == [0, 1]
        checks = {"repeated_mentions": True, "overlap_dedup": True, "boundary": True,
            "flair_whitespace_punctuation_unicode": True, "event_misc": True,
            "nullable_score": True, "zero_mentions": True, "saved_error": True}
    finally:
        DATA, SOURCE_META = original_data, original_meta
    save_json(PATHS["selftests"], checks)
    print(checks)


ERROR:ner_colab_v3:{"model": "spacy", "sample_position": 0, "window": 0, "error": "RuntimeError: intentional test failure"}


{'repeated_mentions': True, 'overlap_dedup': True, 'boundary': True, 'flair_whitespace_punctuation_unicode': True, 'event_misc': True, 'nullable_score': True, 'zero_mentions': True, 'saved_error': True}
SUCCESS 6 — בדיקות חוזה קצרות: completed; 0.00s (0.00min)


7 — BERT
טעינה לפי commit מדויק, fast tokenizer וחלונות חופפים. DATE/TIME/EVENT אינם נתמכים במודל זה; unsupported אינו ציון אפס.

In [22]:
with cell('7 — BERT'):
    name = 'bert'
    revision = hf_revision(name)
    tokenizer = AutoTokenizer.from_pretrained(MODELS[name]["id"], revision=revision, use_fast=True)
    assert tokenizer.is_fast, "A fast tokenizer is required"
    model = AutoModelForTokenClassification.from_pretrained(MODELS[name]["id"], revision=revision)
    ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="average",
                   device=0 if DEVICE.startswith("cuda") else -1)
    limit = min(int(tokenizer.model_max_length), int(model.config.max_position_embeddings))
    print(name, "device:", ner.device, "revision:", revision, "token limit:", limit)
    run_model(name, ner, tokenizer_measure(tokenizer), limit,
              {"revision_model": revision, "revision_tokenizer": revision,
                "labels": sorted(set(model.config.id2label.values())),
                "unsupported": ["DATE", "TIME", "EVENT"], "aggregation_strategy": "average"})
    del ner, model, tokenizer
    release()


bert device: cuda:0 revision: d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc token limit: 512
bert: 10.0% (100/1000); errors=0; 7.3s
bert: 20.0% (200/1000); errors=0; 13.6s
bert: 30.0% (300/1000); errors=0; 22.0s
bert: 40.0% (400/1000); errors=0; 26.1s
bert: 50.0% (500/1000); errors=0; 29.6s
bert: 60.0% (600/1000); errors=0; 31.7s
bert: 70.0% (700/1000); errors=0; 33.6s
bert: 80.0% (800/1000); errors=0; 35.4s
bert: 90.0% (900/1000); errors=0; 37.2s
bert: 100.0% (1000/1000); errors=0; 38.9s
SUCCESS 7 — BERT: completed; 41.98s (0.70min)


8 — RoBERTa
טעינה לפי commit מדויק, fast tokenizer וחלונות חופפים. DATE/TIME/EVENT אינם נתמכים במודל זה; unsupported אינו ציון אפס.

In [23]:
with cell('8 — RoBERTa'):
    name = 'roberta'
    revision = hf_revision(name)
    tokenizer = AutoTokenizer.from_pretrained(MODELS[name]["id"], revision=revision, use_fast=True)
    assert tokenizer.is_fast, "A fast tokenizer is required"
    model = AutoModelForTokenClassification.from_pretrained(MODELS[name]["id"], revision=revision)
    ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="average",
                   device=0 if DEVICE.startswith("cuda") else -1)
    limit = min(int(tokenizer.model_max_length), int(model.config.max_position_embeddings))
    print(name, "device:", ner.device, "revision:", revision, "token limit:", limit)
    run_model(name, ner, tokenizer_measure(tokenizer), limit,
              {"revision_model": revision, "revision_tokenizer": revision,
                "labels": sorted(set(model.config.id2label.values())),
                "unsupported": ["DATE", "TIME", "EVENT"], "aggregation_strategy": "average"})
    del ner, model, tokenizer
    release()


roberta device: cuda:0 revision: 8f3abc1ef81ffbbb0e80568d4fed1dd10d459548 token limit: 512


/usr/local/lib/python3.13/dist-packages/transformers/pipelines/token_classification.py:398: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(


roberta: 10.0% (100/1000); errors=0; 3.2s
roberta: 20.0% (200/1000); errors=0; 6.4s
roberta: 30.0% (300/1000); errors=0; 9.8s
roberta: 40.0% (400/1000); errors=0; 13.3s
roberta: 50.0% (500/1000); errors=0; 16.5s
roberta: 60.0% (600/1000); errors=0; 19.4s
roberta: 70.0% (700/1000); errors=0; 23.3s
roberta: 80.0% (800/1000); errors=0; 26.9s
roberta: 90.0% (900/1000); errors=0; 30.4s
roberta: 100.0% (1000/1000); errors=0; 33.9s
SUCCESS 8 — RoBERTa: completed; 41.09s (0.68min)


9 — spaCy Transformer
spaCy מחזיר score=null. נשמרים כל סוגי האזכורים, כולל EVENT ותוויות שממופות ל־MISC; גרסת החבילה ו־hash של קובצי המודל מחליפים Git revision שאינו זמין לחבילת spaCy.

In [24]:
with cell('9 — spaCy Transformer'):
    if SPACY_DEVICE.startswith("cuda"):
        spacy.require_gpu(0)
    else:
        spacy.require_cpu()
    nlp = spacy.load("en_core_web_trf", disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])
    trf = nlp.get_pipe("transformer").model
    measure_spacy, limit, spacy_actual_device, spacy_backend = spacy_runtime_adapter(nlp)
    def predict_spacy(text):
        return [{"label": e.label_, "text": e.text, "start": e.start_char,
                 "end": e.end_char, "score": None} for e in nlp(text).ents]
    print("spaCy device:", spacy_actual_device, "backend:", spacy_backend, "token limit:", limit)
    run_model("spacy", predict_spacy, measure_spacy, limit,
        {"revision_model": nlp.meta["version"], "revision_tokenizer": None, "device": spacy_actual_device,
         "transformer_backend": spacy_backend,
         "model_files_sha256": file_hashes(nlp.path), "labels": sorted(nlp.get_pipe("ner").labels),
         "spacy_config": nlp.config.to_str(), "unsupported": []})
    del predict_spacy, measure_spacy, trf, nlp
    release()


spaCy device: cpu backend: spacy-curated-transformers token limit: 512
spacy: 10.0% (100/1000); errors=0; 30.2s
spacy: 20.0% (200/1000); errors=0; 57.9s
spacy: 30.0% (300/1000); errors=0; 84.0s
spacy: 40.0% (400/1000); errors=0; 113.6s
spacy: 50.0% (500/1000); errors=0; 141.4s
spacy: 60.0% (600/1000); errors=0; 164.6s
spacy: 70.0% (700/1000); errors=0; 194.1s
spacy: 80.0% (800/1000); errors=0; 224.1s
spacy: 90.0% (900/1000); errors=0; 254.5s
spacy: 100.0% (1000/1000); errors=0; 282.7s
SUCCESS 9 — spaCy Transformer: completed; 289.70s (4.83min)


10 — GLiNER2.5
משתמשים ב־AutoExtractor של gliner2==2.1.0. סופרים את הקידוד כולל כל 22 התוויות, ו־event נכללת במפורש; נשמר פלט גולמי נפרד לכל חלון.

In [25]:
with cell('10 — GLiNER2.5'):
    revision = hf_revision("gliner")
    directory = snapshot_download(MODELS["gliner"]["id"], revision=revision)
    model = AutoExtractor.from_pretrained(directory).to(DEVICE)
    model.eval()
    model.strict_extraction = True  # שגיאות ה־API חייבות להישמר ככשל ולא כפלט ריק
    schema = model.create_schema().entities(GLINER_LABELS)
    limit = int(model.encoder.config.max_position_embeddings)
    def measure_gliner(text):
        # API שנבדק ב־2.1.0: max_len=None מונע חיתוך מילים
        return len(model.processor.transform_record(text, schema, max_len=None).input_ids)
    def predict_gliner(text):
        return model.extract_entities(text, GLINER_LABELS, threshold=GLINER_THRESHOLD,
            include_spans=True, include_confidence=True, max_len=None, overlap_policy="allow")
    assert "event" in GLINER_LABELS
    print("GLiNER device:", next(model.parameters()).device, "revision:", revision, "limit:", limit)
    run_model("gliner", predict_gliner, measure_gliner, limit,
        {"revision_model": revision, "revision_tokenizer": revision,
         "model_files_sha256": file_hashes(directory), "labels": GLINER_LABELS,
         "overlap_policy": "allow", "max_len": None, "unsupported": []})
    del predict_gliner, measure_gliner, schema, model
    release()


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/gliner2/models/boundary/model.py:2212: UserWarning: Checkpoint uses legacy list-valued extra_special_tokens metadata; loading with compatibility normalization.
  tokenizer = load_extractor_tokenizer(tokenizer_source)


GLiNER device: cuda:0 revision: 54785d51df8d86a0d8f2eb212fec15871aad2c4d limit: 512
gliner: 10.0% (100/1000); errors=0; 3.9s
gliner: 20.0% (200/1000); errors=0; 7.7s
gliner: 30.0% (300/1000); errors=0; 11.9s
gliner: 40.0% (400/1000); errors=0; 15.5s
gliner: 50.0% (500/1000); errors=0; 19.0s
gliner: 60.0% (600/1000); errors=0; 23.3s
gliner: 70.0% (700/1000); errors=0; 27.0s
gliner: 80.0% (800/1000); errors=0; 30.8s
gliner: 90.0% (900/1000); errors=0; 35.1s
gliner: 100.0% (1000/1000); errors=0; 38.7s
SUCCESS 10 — GLiNER2.5: completed; 48.73s (0.81min)


11 — Flair ויישור תווים
שומרים את פלט המשפטים ואת מפת הנרמול לכל חלון. שימוש בהקשר חיצוני כבוי בהגדרה גלויה; התקציב נבדק גם לפי טוקניזציית Flair, ולא רק לפי ספירת מילים.

In [26]:
with cell('11 — Flair ויישור תווים'):
    revision = hf_revision("flair")
    model_file = hf_hub_download(MODELS["flair"]["id"], "pytorch_model.bin", revision=revision)
    flair.device = torch.device(DEVICE)
    tagger = SequenceTagger.load(model_file).to(flair.device)
    tagger.eval()
    splitter = SegtokSentenceSplitter()
    embeddings = [e for e in tagger.embeddings.modules() if hasattr(e, "tokenizer") and hasattr(e, "model")]
    if not embeddings:
        raise RuntimeError("Expected transformer embeddings with tokenizer; inspect Flair checkpoint")
    for embedding in embeddings:
        assert hasattr(embedding, "context_length")
        embedding.context_length = 64 if FLAIR_USE_CONTEXT else 0
        assert embedding.context_length == 0, "This window adapter requires FLAIR_USE_CONTEXT=False"
    limit = min(min(int(e.tokenizer.model_max_length), int(e.model.config.max_position_embeddings))
                for e in embeddings)
    def measure_flair(text):
        normalized = collapse(text)[0]
        tokens = [t.text for t in Sentence(normalized)]
        # משאירים מרווח לסימני Flair מעבר לקידוד הבסיסי
        return max(len(e.tokenizer(tokens, is_split_into_words=True,
                   add_special_tokens=True, truncation=False)["input_ids"]) + 8 for e in embeddings)
    def predict_flair(text):
        normalized, starts, ends = collapse(text)
        sentences = splitter.split(normalized)
        records, errors = [], []
        for sentence in sentences:
            record = {"sentence_text": sentence.text,
                "sentence_start_flair": sentence.start_position,
                "entities": [], "flair_output": None, "error": None}
            records.append(record)
            try:
                tagger.predict(sentence, mini_batch_size=1, embedding_storage_mode="none")
                record["entities"] = [{"label": s.get_label("ner").value, "score": float(s.get_label("ner").score),
                              "text": s.text, "start_pos": s.start_position, "end_pos": s.end_position}
                             for s in sentence.get_spans("ner")]
                record["flair_output"] = sentence.to_dict()
            except Exception as exc:
                record["error"] = f"{type(exc).__name__}: {exc}"
                errors.append(record["error"])
        return {"normalized_text": normalized, "source_starts": starts, "source_ends": ends,
                "sentences": records, "errors": errors}
    print("Flair device:", next(tagger.parameters()).device, "revision:", revision, "limit:", limit)
    run_model("flair", predict_flair, measure_flair, limit,
        {"revision_model": revision, "revision_tokenizer": None,
         "checkpoint_sha256": digest_file(model_file),
         "tokenizer_sha256": [digest(e.tokenizer.backend_tokenizer.to_str().encode())
                             if e.tokenizer.is_fast else digest(encoded(e.tokenizer.get_vocab())) for e in embeddings],
         "labels": [str(x) for x in tagger.label_dictionary.get_items()],
         "use_context": FLAIR_USE_CONTEXT, "alignment": "sequential_verified_normalized_positions",
         "embedding_settings": [{"context_length": e.context_length, "stride": e.stride,
               "truncate": e.truncate, "allow_long_sentences": e.allow_long_sentences} for e in embeddings],
         "normalization": "whitespace_collapse_with_character_map", "unsupported": []})
    del predict_flair, measure_flair, embeddings, embedding, tagger, splitter
    release()


2026-10-13 09:57:45,708 SequenceTagger predicts: Dictionary with 76 tags: <unk>, O, B-CARDINAL, E-CARDINAL, S-PERSON, S-CARDINAL, S-PRODUCT, B-PRODUCT, I-PRODUCT, E-PRODUCT, B-WORK_OF_ART, I-WORK_OF_ART, E-WORK_OF_ART, B-PERSON, E-PERSON, S-GPE, B-DATE, I-DATE, E-DATE, S-ORDINAL, S-LANGUAGE, I-PERSON, S-EVENT, S-DATE, B-QUANTITY, E-QUANTITY, S-TIME, B-TIME, I-TIME, E-TIME, B-GPE, E-GPE, S-ORG, I-GPE, S-NORP, B-FAC, I-FAC, E-FAC, B-NORP, E-NORP, S-PERCENT, B-ORG, E-ORG, B-LANGUAGE, E-LANGUAGE, I-CARDINAL, I-ORG, S-WORK_OF_ART, I-QUANTITY, B-MONEY
Flair device: cuda:0 revision: 4ffb3596f4359f0c8799ea15bbf5dbb3b0915a53 limit: 512
flair: 10.0% (100/1000); errors=0; 5.7s
flair: 20.0% (200/1000); errors=0; 12.3s
2026-10-13 09:58:06,899 Warning: An empty Sentence was created! Are there empty strings in your dataset?
flair: 30.0% (300/1000); errors=0; 18.1s
2026-10-13 09:58:17,004 Warning: An empty Sentence was created! Are there empty strings in your dataset?
flair: 40.0% (400/1000); errors=0

12 — פלט מפורט, מועמדים וחמש טבלאות CSV/Parquet
לכל מודל טבלה עצמאית הנגזרת מהאזכורים התקינים שלו, עם מיפוי הסיווגים האחיד. המועמדים הניסיוניים נשמרים בנפרד ואינם קובעים מודל זוכה או משנים את הטבלאות.

In [27]:
with cell('12 — פלט מפורט, מועמדים וחמש טבלאות CSV/Parquet'):
    COLUMNS = ["entity_group", "score", "word", "start", "end", "Source", "Index", "Text", "subtype"]
    ARROW_SCHEMA = pa.schema([(name, pa.int64() if name in {"start", "end", "Index"}
                             else pa.float64() if name == "score" else pa.string()) for name in COLUMNS])
    def table(rows):
        return pa.Table.from_pylist([{k: r.get(k) for k in COLUMNS} for r in rows], schema=ARROW_SCHEMA)
    def roundtrip(csv_path, parquet_path):
        parquet = pq.read_table(parquet_path)
        assert parquet.schema == ARROW_SCHEMA
        frame = pd.read_csv(csv_path, encoding="utf-8-sig", keep_default_na=False,
                            dtype={k: "string" for k in COLUMNS}, float_precision="round_trip")
        assert list(frame.columns) == COLUMNS and len(frame) == parquet.num_rows
        csv_rows = frame.to_dict("records")
        for c, p in zip(csv_rows, parquet.to_pylist()):
            for k in COLUMNS:
                v = c[k]
                if k in {"start", "end", "Index"}:
                    assert int(v) == p[k]
                elif k == "score":
                    assert (v == "" and p[k] is None) or (v != "" and float(v) == p[k])
                else:
                    assert v == ("" if p[k] is None else p[k])
        return parquet.to_pylist()

    ALL = [m for name in MODELS for result in MODEL_OUTPUTS[name]["results"] for m in result["mentions"]]
    save_json(PATHS["mentions_all"], {"schema_version": SCHEMA_VERSION,
        "sample_sha256": SAMPLE_HASH, "mentions": ALL})
    valid = [m for m in ALL if m["status_validation"] == "valid"]
    bucket = defaultdict(list)
    for m in valid:
        key = (m["Source"], m["Index"], m["entity_group"], m["start"], m["end"],
               m["model"] if m["entity_group"] == "EVENT" else None)
        bucket[key].append(m)
    candidates = []
    for i, (key, support) in enumerate(sorted(bucket.items(), key=lambda kv: str(kv[0]))):
        first = support[0]
        candidates.append({"candidate_id": i, **{k: first[k] for k in COLUMNS},
            "sample_position": first["sample_position"], "support": support,
            "supporting_models": sorted({m["model"] for m in support}), "overlapping_alternatives": [],
            "decision": "separate_event_comparison" if key[2] == "EVENT" else "exact_span_candidate",
            "representative_rule": "first in configured model order; no score averaging"})
    by_paragraph = defaultdict(list)
    for c in candidates:
        by_paragraph[(c["Source"], c["Index"])].append(c)
    for values in by_paragraph.values():
        for c in values:
            c["overlapping_alternatives"] = [d["candidate_id"] for d in values if d is not c
                and max(c["start"], d["start"]) < min(c["end"], d["end"])]
            c["selected_for_tables"] = sorted({m["model"] for m in c["support"]
                if TABLE_THRESHOLDS[m["model"]] is None or
                   m["score"] is not None and m["score"] >= TABLE_THRESHOLDS[m["model"]]})
    def iou(a, b):
        intersection = max(0, min(a["end"], b["end"]) - max(a["start"], b["start"]))
        return intersection / (max(a["end"], b["end"]) - min(a["start"], b["start"]))
    date_experiment = []
    if EXPERIMENTAL_DATE:
        primary = [m for m in valid if m["model"] == "flair" and m["entity_group"] == "DATE"]
        additions = [m for m in valid if m["model"] == "gliner" and m["entity_group"] == "DATE"
            and not any(m["Source"] == f["Source"] and m["Index"] == f["Index"]
                        and iou(m, f) >= DATE_OVERLAP_IOU for f in primary)]
        date_experiment = primary + additions
    save_json(PATHS["combination_experimental"], {"schema_version": SCHEMA_VERSION,
        "sample_sha256": SAMPLE_HASH, "experimental": True, "table_derivation": "separate_valid_mentions_per_model",
        "table_models": list(MODELS), "event_auto_union": False, "candidates": candidates,
        "date_experiment_enabled": EXPERIMENTAL_DATE, "date_iou_threshold": DATE_OVERLAP_IOU,
        "date_experiment": date_experiment})
    def select_table(mentions, name):
        rows, seen = [], set()
        for m in mentions:
            if m["model"] != name or m["status_validation"] != "valid":
                continue
            threshold = TABLE_THRESHOLDS[name]
            if threshold is not None and (m["score"] is None or m["score"] < threshold):
                continue
            key = (m["Source"], m["Index"], m["entity_group"], m["start"], m["end"])
            if key not in seen:
                rows.append(m)
                seen.add(key)
        return rows
    TABLE_ROWS = {}
    for name in MODELS:
        rows = select_table(valid, name)
        rows.sort(key=lambda m: (m["Source"], m["Index"], m["start"], m["end"], m["entity_group"]))
        TABLE_ROWS[name] = rows
        arrow = table(rows)
        with PATHS[f"entities_table_{name}_csv"].open("x", encoding="utf-8-sig", newline="") as handle:
            arrow.to_pandas().to_csv(handle, index=False, columns=COLUMNS)
        with PATHS[f"entities_table_{name}_parquet"].open("xb") as handle:
            pq.write_table(arrow, handle)
        assert roundtrip(PATHS[f"entities_table_{name}_csv"], PATHS[f"entities_table_{name}_parquet"]) == arrow.to_pylist()
        print(f"{name}: {len(rows)} table rows; CSV and Parquet verified")
    # round-trip נוסף עם null, EVENT, MISC ו־CSV escaping, גם אם המדגם האמיתי ריק
    with tempfile.TemporaryDirectory() as tmp:
        test_rows = [dict(entity_group=g, score=None, word='A,"B"', start=0, end=5,
                         Source="test", Index=0, Text='A,"B"', subtype="weapon" if g=="MISC" else "EVENT")
                     for g in ["EVENT", "MISC"]]
        t = table(test_rows)
        cp, pp = Path(tmp)/"test.csv", Path(tmp)/"test.parquet"
        t.to_pandas().to_csv(cp, index=False, encoding="utf-8-sig")
        pq.write_table(t, pp)
        assert roundtrip(cp, pp) == test_rows
    print("Detailed mentions:", len(ALL), "Candidates:", len(candidates), "Table rows:", {name: len(rows) for name, rows in TABLE_ROWS.items()})


bert: 7085 table rows; CSV and Parquet verified
roberta: 6154 table rows; CSV and Parquet verified
spacy: 8734 table rows; CSV and Parquet verified
gliner: 9959 table rows; CSV and Parquet verified
flair: 8947 table rows; CSV and Parquet verified
Detailed mentions: 41550 Candidates: 18984 Table rows: {'bert': 7085, 'roberta': 6154, 'spacy': 8734, 'gliner': 9959, 'flair': 8947}
SUCCESS 12 — פלט מפורט, מועמדים וחמש טבלאות CSV/Parquet: completed; 8.11s (0.14min)


13 — אימות קבצים, סיכום ומניפסט
פותחים מחדש כל פלט ובודקים את המקור, המדגם, התמיכה במועמדים, היסטים, שגיאות ותוויות. success ניתן רק כשאין כשלים או אזכורים לא תקינים.

In [28]:
with cell('13 — אימות קבצים, סיכום ומניפסט'):
    assert digest(PATHS["source"].read_bytes()) == SOURCE_META["sha256"]
    saved_data = load_json(PATHS["source"])
    saved_sample = load_json(PATHS["sample"])
    assert digest(PATHS["sample"].read_bytes()) == SAMPLE_HASH
    assert saved_sample["paragraphs"] == SAMPLE and len(SAMPLE) == SAMPLE_SIZE
    assert saved_sample["split_version"] == SPLIT_VERSION
    saved_models, summary, reconstructed = {}, {}, []
    for name in MODELS:
        output = load_json(PATHS["ner_"+name])
        saved_models[name] = output
        assert output["sample_sha256"] == SAMPLE_HASH and output["source"] == SOURCE_META
        assert len(output["results"]) == SAMPLE_SIZE
        counts = Counter()
        labels = defaultdict(Counter)
        event_paragraphs = set()
        audit_started, audit_last, audit_errors = time.perf_counter(), 0, 0
        for i, result in enumerate(output["results"]):
            p = result["paragraph"]
            assert p == SAMPLE[i] and p["sample_position"] == i
            assert saved_data[p["Source"]][p["start_document"]:p["end_document"]] == p["Text"]
            status = result["status_processing"]
            assert status in {"success", "partial_invalid", "error"}
            counts[status] += 1
            if status != "success":
                assert result["errors"]
            for w in result["raw"]:
                assert p["Text"][w["start"]:w["end"]] == w["Text"]
                assert w["encoded_tokens"] <= min(WINDOWS[name]["max_encoded_tokens"], output["metadata"]["effective_encoded_limit"])
                if w["status_processing"] == "error":
                    assert w["error"]
            if status == "success" and not result["mentions"]:
                counts["zero_mentions"] += 1
            for m in result["mentions"]:
                assert m["status_validation"] in {"valid", "invalid", "pending"}
                counts[m["status_validation"]+"_mentions"] += 1
                labels[m["label_original"]][m["status_validation"]] += 1
                if m["status_validation"] == "valid":
                    assert p["Text"][m["start"]:m["end"]] == m["word"]
                    assert m["start_document"] == p["start_document"]+m["start"]
                    assert m["end_document"] == p["start_document"]+m["end"]
                    assert saved_data[p["Source"]][m["start_document"]:m["end_document"]] == m["word"]
                    if m["entity_group"] == "EVENT":
                        event_paragraphs.add(i)
                else:
                    assert m["error"]
                for ref in m["ref_output_raw"]:
                    filename, pointer = ref.split("#", 1)
                    assert filename == PATHS["ner_"+name].name
                    node = output
                    for token in pointer.strip("/").split("/"):
                        node = node[int(token)] if isinstance(node, list) else node[token]
                    assert node is not None
                reconstructed.append(m)
            audit_errors += status != "success"
            audit_last = progress("audit_"+name, i+1, SAMPLE_SIZE, audit_errors, audit_started, audit_last)
        for key in ["success", "partial_invalid", "error", "zero_mentions",
                    "valid_mentions", "invalid_mentions", "pending_mentions"]:
            counts.setdefault(key, 0)
        summary[name] = {**dict(counts), "paragraphs_succeeded": counts["success"],
            "paragraphs_failed_or_invalid": counts["error"]+counts["partial_invalid"],
            "labels": {k: dict(v) for k, v in labels.items()}, "event_paragraphs": len(event_paragraphs),
            "processing_seconds": output["processing_seconds"], "unsupported": output["metadata"]["unsupported"]}
    assert load_json(PATHS["mentions_all"])["mentions"] == reconstructed
    saved_combo = load_json(PATHS["combination_experimental"])
    assert saved_combo["sample_sha256"] == SAMPLE_HASH and saved_combo["event_auto_union"] is False
    supports = [m for c in saved_combo["candidates"] for m in c["support"]]
    assert sorted(map(encoded, supports)) == sorted(map(encoded, [m for m in reconstructed if m["status_validation"]=="valid"]))
    for c in saved_combo["candidates"]:
        assert c["supporting_models"] == sorted({m["model"] for m in c["support"]})
        assert all((m["Source"],m["Index"],m["entity_group"],m["start"],m["end"]) ==
                   (c["Source"],c["Index"],c["entity_group"],c["start"],c["end"]) for m in c["support"])
        if c["entity_group"] == "EVENT":
            assert len(c["supporting_models"]) == 1
    for group_name in ["EVENT", "MISC"]:
        emitted = [m for m in reconstructed if m["status_validation"]=="valid" and m["entity_group"]==group_name]
        assert sum(m["entity_group"]==group_name for m in supports) == len(emitted)
    TABLE_COUNTS = {}
    for name in MODELS:
        actual_rows = roundtrip(PATHS[f"entities_table_{name}_csv"], PATHS[f"entities_table_{name}_parquet"])
        assert actual_rows == table(TABLE_ROWS[name]).to_pylist()
        expected_rows = select_table(reconstructed, name)
        assert sorted(map(encoded, table(expected_rows).to_pylist())) == sorted(map(encoded, actual_rows))
        for group_name in ["EVENT", "MISC"]:
            expected = sum(m["entity_group"] == group_name for m in expected_rows)
            assert sum(m["entity_group"] == group_name for m in actual_rows) == expected
        for row in actual_rows:
            p = next(p for p in SAMPLE if (p["Source"], p["Index"]) == (row["Source"], row["Index"]))
            assert row["Text"] == p["Text"] and row["Text"][row["start"]:row["end"]] == row["word"]
        TABLE_COUNTS[name] = len(actual_rows)
        summary[name]["table_rows"] = len(actual_rows)
        summary[name]["table_files"] = {ext: PATHS[f"entities_table_{name}_{ext}"].name for ext in ["csv", "parquet"]}
    RUN_STATUS = "completed_with_errors" if any(
        s.get("error",0) or s.get("partial_invalid",0) or s.get("pending_mentions",0)
        for s in summary.values()) else "success"
    save_json(PATHS["validation_summary"], {"schema_version": SCHEMA_VERSION,
        "run_status": RUN_STATUS, "models": summary, "sample_size": SAMPLE_SIZE,
        "table_rows": TABLE_COUNTS, "sample_sha256": SAMPLE_HASH,
        "checks": ["source_hash", "sample_identity", "all_models_same_sample", "exact_offsets",
                   "raw_references", "candidate_support", "EVENT_MISC_retained", "csv_parquet_roundtrip"],
        "selftests": load_json(PATHS["selftests"]), "roundtrip_selftest": True})
    required = {p.name for key, p in PATHS.items() if key != "manifest"}
    assert {p.name for p in RUN_DIR.iterdir()} == required
    manifest = {"schema_version": SCHEMA_VERSION, "run_name": RUN_NAME, "run_status": RUN_STATUS,
        "at_processed": utc_now(), "date_jerusalem": YYYYMMDD, "source": SOURCE_META,
        "sample_sha256": SAMPLE_HASH, "configuration": CONFIG, "configuration_sha256": CONFIG_HASH,
        "versions": VERSIONS, "python": platform.python_version(), "platform": platform.platform(),
        "cuda": torch.version.cuda, "models": {k: v["metadata"] for k,v in saved_models.items()},
        "summary": summary, "files_sha256": {p.name: digest(p.read_bytes())
              for key,p in PATHS.items() if key != "manifest"},
        "manifest_hash_policy": "Manifest excluded from its own hashes; ZIP checks exact manifest bytes"}
    manifest["dependency_checks"] = DEPENDENCY_REPORT
    manifest["timing_files"] = {key: path.name for key, path in TIMING_PATHS.items()}
    save_json(PATHS["manifest"], manifest)
    print("Run status:", RUN_STATUS)
    for name, values in summary.items():
        print(name, json.dumps(values, ensure_ascii=False))


audit_bert: 10.0% (100/1000); errors=0; 0.0s
audit_bert: 20.0% (200/1000); errors=0; 0.0s
audit_bert: 30.0% (300/1000); errors=0; 0.0s
audit_bert: 40.0% (400/1000); errors=0; 0.0s
audit_bert: 50.0% (500/1000); errors=0; 0.0s
audit_bert: 60.0% (600/1000); errors=0; 0.0s
audit_bert: 70.0% (700/1000); errors=0; 0.0s
audit_bert: 80.0% (800/1000); errors=0; 0.0s
audit_bert: 90.0% (900/1000); errors=0; 0.1s
audit_bert: 100.0% (1000/1000); errors=0; 0.1s
audit_roberta: 10.0% (100/1000); errors=0; 0.0s
audit_roberta: 20.0% (200/1000); errors=0; 0.0s
audit_roberta: 30.0% (300/1000); errors=0; 0.0s
audit_roberta: 40.0% (400/1000); errors=0; 0.0s
audit_roberta: 50.0% (500/1000); errors=0; 0.0s
audit_roberta: 60.0% (600/1000); errors=0; 0.0s
audit_roberta: 70.0% (700/1000); errors=0; 0.0s
audit_roberta: 80.0% (800/1000); errors=0; 0.0s
audit_roberta: 90.0% (900/1000); errors=0; 0.0s
audit_roberta: 100.0% (1000/1000); errors=0; 0.1s
audit_spacy: 10.0% (100/1000); errors=0; 0.0s
audit_spacy: 20.0% (

14 — יצירת ZIP ואימות
הארכיון מכיל רק את תיקיית ההרצה הנוכחית. נבדקים שמות הקבצים, CRC וכל ה־hashes; בתא הבא יתווספו זמני התאים ותופעל ההורדה.

In [29]:
with cell('14 — יצירת ZIP ואימות'):
    if ZIP_PATH.exists():
        raise FileExistsError(f"ZIP already exists: {ZIP_PATH}")
    saved_manifest = load_json(PATHS["manifest"])
    assert saved_manifest == manifest
    members = {f"{RUN_NAME}/{p.name}": p for p in PATHS.values()}
    assert set(p.name for p in RUN_DIR.iterdir()) == set(p.name for p in PATHS.values())
    with zipfile.ZipFile(ZIP_PATH, "x", compression=zipfile.ZIP_DEFLATED) as archive:
        for archive_name, path in members.items():
            archive.write(path, archive_name)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        assert archive.testzip() is None
        assert set(archive.namelist()) == set(members) and len(archive.namelist()) == len(members)
        for archive_name, path in members.items():
            archived = archive.read(archive_name)
            assert archived == path.read_bytes()
            if path != PATHS["manifest"]:
                assert digest(archived) == saved_manifest["files_sha256"][path.name]
        assert json.loads(archive.read(f"{RUN_NAME}/{PATHS['sample'].name}"))["paragraphs"] == SAMPLE
        for name in MODELS:
            parquet_member = f"{RUN_NAME}/{PATHS[f'entities_table_{name}_parquet'].name}"
            assert pq.read_table(io.BytesIO(archive.read(parquet_member))).to_pylist() == table(TABLE_ROWS[name]).to_pylist()
    print("Run status:", RUN_STATUS, "ZIP SHA256:", digest(ZIP_PATH.read_bytes()))
    print("ZIP:", ZIP_PATH)
    print("\n".join(sorted(members)))


Run status: completed_with_errors ZIP SHA256: 99b5f5b19718d6e886eae70b029a8c71d16de4e096c70e51d156b9643dbc199b
ZIP: /content/TMP_NER/ner_seed17_n1000_r915351843_20261013.zip
ner_seed17_n1000_r915351843_20261013/combination_experimental_ner_seed17_n1000_r915351843_20261013.json
ner_seed17_n1000_r915351843_20261013/entities_table_bert_ner_seed17_n1000_r915351843_20261013.csv
ner_seed17_n1000_r915351843_20261013/entities_table_bert_ner_seed17_n1000_r915351843_20261013.parquet
ner_seed17_n1000_r915351843_20261013/entities_table_flair_ner_seed17_n1000_r915351843_20261013.csv
ner_seed17_n1000_r915351843_20261013/entities_table_flair_ner_seed17_n1000_r915351843_20261013.parquet
ner_seed17_n1000_r915351843_20261013/entities_table_gliner_ner_seed17_n1000_r915351843_20261013.csv
ner_seed17_n1000_r915351843_20261013/entities_table_gliner_ner_seed17_n1000_r915351843_20261013.parquet
ner_seed17_n1000_r915351843_20261013/entities_table_roberta_ner_seed17_n1000_r915351843_20261013.csv
ner_seed17_n100

15 — סיכום זמני התאים והורדה
זמני כל 15 מקטעי העיבוד, כולל ניסיונות שנכשלו, נרשמים ב־JSON וב־CSV ונוספים ל־ZIP עם hashes. כתיבת הלוג עצמו והפעלת ההורדה מתבצעות אחרי עצירת הטיימר כדי לא ליצור מדידה מעגלית.

In [30]:
with cell('15 — סיכום זמני התאים והורדה'):
    assert ZIP_PATH.is_file()
    with zipfile.ZipFile(ZIP_PATH) as archive:
        assert archive.testzip() is None
        assert len(archive.namelist()) == len(PATHS)
    print("All model outputs verified; preparing the complete cell timing log.")
try:
    print_cell_timings()
    save_json(TIMING_PATHS["cell_timings"], {"schema_version": SCHEMA_VERSION,
        "run_name": RUN_NAME, "measurement_scope": "timed code blocks; excludes timing-log bookkeeping and browser download",
        "total_seconds": sum(row["seconds"] for row in CELL_TIMINGS), "cells": CELL_TIMINGS})
    with TIMING_PATHS["cell_timings_table"].open("x", encoding="utf-8-sig", newline="") as handle:
        pd.DataFrame(CELL_TIMINGS).to_csv(handle, index=False)
    timing_hashes = {path.name: digest_file(path) for key, path in TIMING_PATHS.items() if key != "timing_manifest"}
    save_json(TIMING_PATHS["timing_manifest"], {"schema_version": SCHEMA_VERSION,
        "run_name": RUN_NAME, "files_sha256": timing_hashes,
        "measurement_scope": "durations exclude this timing-log publication and browser transfer"})
    timing_members = {f"{RUN_NAME}/{path.name}": path for path in TIMING_PATHS.values()}
    with zipfile.ZipFile(ZIP_PATH) as archive:
        assert not set(archive.namelist()) & set(timing_members), "Timing log already exists in ZIP"
    with zipfile.ZipFile(ZIP_PATH, "a", compression=zipfile.ZIP_DEFLATED) as archive:
        for member, path in timing_members.items():
            archive.write(path, member)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        assert archive.testzip() is None
        assert set(archive.namelist()) == set(members) | set(timing_members)
        assert len(archive.namelist()) == len(members) + len(timing_members)
        for member, path in {**members, **timing_members}.items():
            assert archive.read(member) == path.read_bytes()
        ledger = json.loads(archive.read(f"{RUN_NAME}/{TIMING_PATHS['timing_manifest'].name}"))
        for filename, expected in {**saved_manifest["files_sha256"], **ledger["files_sha256"]}.items():
            assert digest(archive.read(f"{RUN_NAME}/{filename}")) == expected
        saved_timings = json.loads(archive.read(f"{RUN_NAME}/{TIMING_PATHS['cell_timings'].name}"))
        assert saved_timings["cells"] == CELL_TIMINGS
    print("SUCCESS timing-log publication and final ZIP verification")
    print("Final ZIP SHA256:", digest_file(ZIP_PATH))
    files.download(str(ZIP_PATH))
except BaseException as exc:
    print(f"FAILED 15 — timing-log publication/download: {type(exc).__name__}: {exc}", flush=True)
    raise


All model outputs verified; preparing the complete cell timing log.
SUCCESS 15 — סיכום זמני התאים והורדה: completed; 0.64s (0.01min)

CELL TIMINGS — measured code blocks (logging/download bookkeeping excluded)
1 — התקנות | 92.69s | 1.54min | success
2 — ייבואים וגרסאות | 27.36s | 0.46min | success
3 — הגדרות מרכזיות | 0.00s | 0.00min | success
4 — כלי מקור, חלונות, היסטים ושמירה | 0.00s | 0.00min | success
5 — מקור GitHub ומדגם | 17.30s | 0.29min | success
6 — בדיקות חוזה קצרות | 0.00s | 0.00min | success
7 — BERT | 11.64s | 0.19min | success
8 — RoBERTa | 21.10s | 0.35min | success
9 — spaCy Transformer | 84.83s | 1.41min | success
10 — GLiNER2.5 | 33.25s | 0.55min | success
11 — Flair ויישור תווים | 109.88s | 1.83min | success
12 — פלט מפורט, מועמדים וחמש טבלאות CSV/Parquet | 1.84s | 0.03min | success
13 — אימות קבצים, סיכום ומניפסט | 3.12s | 0.05min | success
14 — יצירת ZIP ואימות | 2.53s | 0.04min | success
15 — סיכום זמני התאים והורדה | 0.16s | 0.00min | success
1 — התקנות | 69.22

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


### הקבצים הצפויים ב־ZIP והרצת Run all

כל שם להלן מקבל את הסיומת `_ner_seed{SEED}_n{SAMPLE_SIZE}_r{RUN_RANDOM_ID}_{YYYYMMDD}`:

- source.json — הבתים המקוריים של DATA.json.
- sample.json — המדגם והיסטי הפסקאות.
- ner_bert.json, ner_roberta.json, ner_spacy.json, ner_gliner.json, ner_flair.json.
- mentions_all.json — אזכורים מפורטים, כולל invalid ושגיאות.
- combination_experimental.json — תמיכות, חלופות חופפות והשוואת EVENT נפרדת.
- entities_table_bert.csv, entities_table_bert.parquet.
- entities_table_roberta.csv, entities_table_roberta.parquet.
- entities_table_spacy.csv, entities_table_spacy.parquet.
- entities_table_gliner.csv, entities_table_gliner.parquet.
- entities_table_flair.csv, entities_table_flair.parquet.
- selftests.json, environment.txt, validation_summary.json, manifest.json.
- installation_log.txt — פלטי ההתקנות ובדיקות התלויות.
- cell_timings.json, cell_timings_table.csv — כותרת כל תא, זמן בשניות ובדקות, סטטוס, שגיאה וחותמות UTC.
- timing_manifest.json — hashes של קובצי הזמנים; מופק אחרי שהתא האחרון נמדד.

פתחו את המחברת ב־Colab, בחרו runtime חדש עם GPU, ערכו את תא 3 ובחרו Runtime → Run all. אין להעלות את CSV הדוגמה. בסיום תא 15 מוצג סיכום זמני התאים ויורד ZIP אחד. הזמן הוא משך מקטע העיבוד בתוך with cell; כתיבת לוג הזמנים עצמו, הוספתו לארכיון והעברת ההורדה בדפדפן אינן נכללות במדידה כדי למנוע מדידה מעגלית. אם תא נכשל, מוצגים זמני כל התאים שבוצעו עד אליו והחריגה עולה מחדש. אפשר להציג את הלוג שבזיכרון בכל שלב באמצעות print_cell_timings(). ניסיונות חוזרים נשמרים בנפרד; runtime חדש מתחיל לוג חדש. לשחזור הציבו SOURCE_COMMIT ואת revisions של המודלים מהמניפסט, ושמרו על גרסאות הספריות; בחרו RUN_RANDOM_ID חדש כדי למנוע דריסה. אותו מקור/SEED/SAMPLE_SIZE ייתן אותו מדגם, אך חישוב GPU אינו הבטחה לתוצאות bit-for-bit זהות. כשל פסקה נשמר ונותן completed_with_errors; כשל התקנה, טעינה או שמירה עוצר. ההורדה בדפדפן עשויה לבקש אישור; messages של SUCCESS מתייחסים להשלמת התא, ו־run_status מתייחס לשלמות תוצאות ההרצה.
